# 36. 市場データの検査とAPI接続
出典：FX (3).ipynb、元セルindex [87, 88, 89]。保存出力は results/imported_fx3/。
研究履歴の原本です。Notebookの変数・価格CSV・学習済みファイルに依存します。
失敗した試行も保管しています。一括実行やAPI接続を開始する入口ではありません。
元コード内の指示・自動判定名は資料として保存しています。独立した検証済みの結論とは区別してください。


## 元セルindex 87
構文状態：valid


In [ ]:
# ============================================================
# FORWARD PAPER DATA GATE
# Real closed USDJPY 15m bars -> Frozen Forward Paper Engine
#
# PURPOSE
#   1. Read REAL incoming 15m OHLC bars
#   2. Validate timestamps / OHLC / duplicates / closure
#   3. Prevent accidental history gaps
#   4. Feed bars sequentially into run_forward_paper_once()
#   5. Keep an audit trail
#
# IMPORTANT
#   - NO strategy optimization
#   - NO model retraining
#   - NO synthetic market bars
#   - NO live broker orders
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import inspect
import traceback

print("=" * 90)
print("FORWARD PAPER DATA GATE")
print("=" * 90)


# ============================================================
# 1. BASIC HELPERS
# ============================================================

REQUIRED_COLS = ["timestamp", "open", "high", "low", "close"]

def _safe_timestamp(x):
    """Convert anything timestamp-like to UTC Timestamp."""
    try:
        return pd.to_datetime(x, utc=True)
    except Exception:
        return pd.NaT


def _get_time_series(df):
    """
    Return UTC timestamp Series from dataframe.
    Accept timestamp either as column or DatetimeIndex.
    """
    if "timestamp" in df.columns:
        return pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

    if isinstance(df.index, pd.DatetimeIndex):
        return pd.Series(
            pd.to_datetime(df.index, utc=True, errors="coerce"),
            index=df.index
        )

    return pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")


def _extract_existing_timestamps(df):
    """
    Extract already-processed timestamps from a dataframe if possible.
    """
    if df is None or not isinstance(df, pd.DataFrame) or len(df) == 0:
        return set()

    candidates = [
        "timestamp",
        "signal_time",
        "bar_time",
        "_diag_signal_time",
    ]

    for col in candidates:
        if col in df.columns:
            s = pd.to_datetime(df[col], utc=True, errors="coerce")
            return set(s.dropna().tolist())

    if isinstance(df.index, pd.DatetimeIndex):
        return set(
            pd.to_datetime(df.index, utc=True, errors="coerce")
            .dropna()
            .tolist()
        )

    return set()


def _allowed_weekend_gap(t1, t2):
    """
    Allow normal FX weekend discontinuity.

    We do NOT attempt to invent missing bars.
    A gap is only automatically tolerated when it starts Friday
    and resumes Sunday/Monday.
    """
    if pd.isna(t1) or pd.isna(t2):
        return False

    wd1 = t1.weekday()
    wd2 = t2.weekday()

    return (
        wd1 == 4 and wd2 in (6, 0)
    )


# ============================================================
# 2. FIND FROZEN CHAMPION DIRECTORY
# ============================================================

champion_candidates = []

root = Path("production_champion")

if root.exists():
    for p in root.glob("champion_v1_base_plus_regime_*"):
        if p.is_dir():
            champion_candidates.append(p)

if len(champion_candidates) == 0:
    print()
    print("STOP:")
    print("Frozen Champion directory was not found.")
    print("Expected something like:")
    print(
        r"production_champion\champion_v1_base_plus_regime_20260909"
    )

    FORWARD_DATA_GATE_READY = False

else:
    champion_dir = sorted(
        champion_candidates,
        key=lambda p: p.name
    )[-1]

    print("Champion directory:", champion_dir)


# ============================================================
# 3. CHECK FROZEN LIVE / PAPER ENGINE
# ============================================================

if len(champion_candidates) > 0:

    engine_exists = (
        "run_forward_paper_once" in globals()
        and callable(globals()["run_forward_paper_once"])
    )

    print("run_forward_paper_once exists:", engine_exists)

    if not engine_exists:
        print()
        print("STOP:")
        print(
            "run_forward_paper_once() is not currently loaded "
            "in this Jupyter kernel."
        )
        print(
            "Run the previous Forward Paper Trading build cell first."
        )

        FORWARD_DATA_GATE_READY = False


# ============================================================
# 4. RESOLVE PAPER FORWARD DIRECTORY
# ============================================================

if len(champion_candidates) > 0 and engine_exists:

    paper_dir = champion_dir / "runtime" / "paper_forward"
    paper_dir.mkdir(parents=True, exist_ok=True)

    incoming_path = paper_dir / "incoming_usdjpy_15m.csv"
    feed_path = paper_dir / "usdjpy_15m_forward_feed.csv"
    audit_path = paper_dir / "forward_data_gate_audit.csv"

    # Create empty REAL-data input template only if absent.
    # No fake price row is created.
    if not incoming_path.exists():
        pd.DataFrame(columns=REQUIRED_COLS).to_csv(
            incoming_path,
            index=False
        )

    print()
    print("=" * 90)
    print("FILES")
    print("=" * 90)
    print("Incoming real bars :", incoming_path)
    print("Forward feed       :", feed_path)
    print("Audit log          :", audit_path)


# ============================================================
# 5. DETERMINE CURRENT CANONICAL CUTOFF
# ============================================================

canonical_df = None
canonical_source = None

if len(champion_candidates) > 0 and engine_exists:

    canonical_names = [
        "PRODUCTION_CANONICAL_HISTORY",
        "PRODUCTION_CANONICAL_BARS",
        "CANONICAL_HISTORY",
        "bars_rebuilt_clean",
        "bars",
    ]

    for name in canonical_names:
        obj = globals().get(name)

        if isinstance(obj, pd.DataFrame) and len(obj) > 0:
            ts = _get_time_series(obj)

            if ts.notna().sum() > 0:
                canonical_df = obj
                canonical_source = name
                break

    if canonical_df is None:
        print()
        print("STOP:")
        print("Canonical historical dataframe was not found.")
        print(
            "The frozen live engine must be loaded before "
            "Forward Paper ingestion."
        )

        FORWARD_DATA_GATE_READY = False

    else:
        canonical_times = _get_time_series(canonical_df)
        canonical_latest = canonical_times.max()

        print()
        print("=" * 90)
        print("CANONICAL HISTORY")
        print("=" * 90)
        print("Source       :", canonical_source)
        print("Rows         :", len(canonical_df))
        print("Latest bar   :", canonical_latest)


# ============================================================
# 6. FIND ALREADY PROCESSED FORWARD BARS
# ============================================================

processed_times = set()

if (
    len(champion_candidates) > 0
    and engine_exists
    and canonical_df is not None
):

    # Existing engine feed
    if feed_path.exists():
        try:
            feed_df = pd.read_csv(feed_path)
            processed_times |= _extract_existing_timestamps(feed_df)
        except Exception:
            pass

    # Our own gate audit
    if audit_path.exists():
        try:
            old_audit = pd.read_csv(audit_path)

            if (
                "timestamp" in old_audit.columns
                and "status" in old_audit.columns
            ):
                good = old_audit[
                    old_audit["status"].isin(
                        ["ENGINE_OK", "ALREADY_PROCESSED"]
                    )
                ]

                processed_times |= set(
                    pd.to_datetime(
                        good["timestamp"],
                        utc=True,
                        errors="coerce"
                    )
                    .dropna()
                    .tolist()
                )
        except Exception:
            pass

    if len(processed_times):
        last_processed = max(processed_times)
    else:
        last_processed = canonical_latest

    print("Last processed:", last_processed)


# ============================================================
# 7. REAL-BAR VALIDATOR
# ============================================================

def validate_forward_bars(new_bars, reference_time=None):
    """
    Validate user/API supplied REAL 15m bars.

    Returns:
        cleaned_df,
        error_messages,
        warning_messages
    """

    errors = []
    warnings = []

    if not isinstance(new_bars, pd.DataFrame):
        return (
            pd.DataFrame(columns=REQUIRED_COLS),
            ["Input must be a pandas DataFrame."],
            []
        )

    df = new_bars.copy()

    # Remove accidental CSV index columns.
    df = df[
        [
            c for c in df.columns
            if not str(c).lower().startswith("unnamed")
        ]
    ]

    missing = [c for c in REQUIRED_COLS if c not in df.columns]

    if missing:
        errors.append(
            f"Missing required columns: {missing}"
        )
        return (
            pd.DataFrame(columns=REQUIRED_COLS),
            errors,
            warnings
        )

    df = df[REQUIRED_COLS].copy()

    # Timestamp -> UTC
    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        utc=True,
        errors="coerce"
    )

    if df["timestamp"].isna().any():
        errors.append(
            f"Invalid timestamps: {int(df['timestamp'].isna().sum())}"
        )

    # Numeric OHLC
    for col in ["open", "high", "low", "close"]:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    bad_numeric = df[
        ["open", "high", "low", "close"]
    ].isna().any(axis=1)

    if bad_numeric.any():
        errors.append(
            f"Invalid OHLC rows: {int(bad_numeric.sum())}"
        )

    # Remove fully invalid rows only after recording error.
    df = df.dropna(
        subset=REQUIRED_COLS
    ).copy()

    if len(df) == 0:
        return df, errors, warnings

    # Positive prices
    nonpositive = (
        df[["open", "high", "low", "close"]] <= 0
    ).any(axis=1)

    if nonpositive.any():
        errors.append(
            f"Non-positive price rows: {int(nonpositive.sum())}"
        )

    # OHLC consistency
    high_invalid = (
        df["high"]
        <
        df[["open", "close"]].max(axis=1)
    )

    low_invalid = (
        df["low"]
        >
        df[["open", "close"]].min(axis=1)
    )

    range_invalid = df["high"] < df["low"]

    n_ohlc_bad = int(
        (
            high_invalid
            | low_invalid
            | range_invalid
        ).sum()
    )

    if n_ohlc_bad:
        errors.append(
            f"OHLC consistency failures: {n_ohlc_bad}"
        )

    # 15-minute grid
    grid_ok = (
        df["timestamp"].dt.minute.isin(
            [0, 15, 30, 45]
        )
        &
        (df["timestamp"].dt.second == 0)
        &
        (df["timestamp"].dt.microsecond == 0)
    )

    if (~grid_ok).any():
        errors.append(
            f"Off-grid timestamps: {int((~grid_ok).sum())}"
        )

    # Duplicate timestamps
    duplicate_count = int(
        df["timestamp"].duplicated().sum()
    )

    if duplicate_count:
        errors.append(
            f"Duplicate timestamps in incoming data: "
            f"{duplicate_count}"
        )

    # Sort
    df = (
        df.sort_values("timestamp")
        .reset_index(drop=True)
    )

    # A bar timestamp is BAR OPEN time.
    # It is considered closed 15 minutes later.
    now_utc = pd.Timestamp.now(tz="UTC")

    not_closed = (
        df["timestamp"]
        + pd.Timedelta(minutes=15)
        >
        now_utc
    )

    if not_closed.any():
        errors.append(
            f"Still-open / future bars: "
            f"{int(not_closed.sum())}"
        )

    # Must be newer than reference
    if reference_time is not None:
        old_or_equal = (
            df["timestamp"]
            <= reference_time
        )

        if old_or_equal.any():
            warnings.append(
                f"{int(old_or_equal.sum())} row(s) are "
                "already historical/processed and will be skipped."
            )

    return df, errors, warnings


# ============================================================
# 8. CONTINUITY CHECK
# ============================================================

def check_forward_continuity(df, previous_timestamp):
    """
    Reject unexplained missing weekday 15m bars.

    Weekend gaps are allowed.
    """

    problems = []

    if len(df) == 0:
        return problems

    times = df["timestamp"].tolist()

    previous = previous_timestamp

    for current in times:

        if previous is None:
            previous = current
            continue

        gap = current - previous

        if gap == pd.Timedelta(minutes=15):
            previous = current
            continue

        if gap <= pd.Timedelta(0):
            problems.append(
                {
                    "previous": previous,
                    "current": current,
                    "gap": str(gap),
                    "reason": "NON_INCREASING_TIME",
                }
            )

        elif _allowed_weekend_gap(previous, current):
            # Normal FX weekend closure accepted.
            pass

        else:
            problems.append(
                {
                    "previous": previous,
                    "current": current,
                    "gap": str(gap),
                    "reason": "UNEXPLAINED_MARKET_DATA_GAP",
                }
            )

        previous = current

    return problems


# ============================================================
# 9. AUDIT LOGGER
# ============================================================

def append_gate_audit(records):
    if not records:
        return

    audit_df = pd.DataFrame(records)

    if audit_path.exists():
        try:
            old = pd.read_csv(audit_path)
            audit_df = pd.concat(
                [old, audit_df],
                ignore_index=True
            )
        except Exception:
            pass

    audit_df.to_csv(
        audit_path,
        index=False
    )


# ============================================================
# 10. SEQUENTIAL REAL-BAR INGESTION
# ============================================================

def run_real_forward_bars(real_bars):
    """
    Safely pass REAL, CLOSED 15m USDJPY bars one-by-one
    into the already frozen Forward Paper Engine.

    This wrapper NEVER sends live broker orders.
    """

    global processed_times

    print()
    print("=" * 90)
    print("REAL FORWARD BAR INGESTION")
    print("=" * 90)

    reference = (
        max(processed_times)
        if len(processed_times)
        else canonical_latest
    )

    df, errors, warnings = validate_forward_bars(
        real_bars,
        reference_time=reference
    )

    for w in warnings:
        print("WARNING:", w)

    if errors:
        print()
        print("STOP - DATA VALIDATION FAILED")
        for e in errors:
            print(" -", e)

        return {
            "passed": False,
            "reason": "DATA_VALIDATION_FAILED",
            "errors": errors,
        }

    # Skip already-known timestamps.
    df = df[
        ~df["timestamp"].isin(processed_times)
    ].copy()

    # Also reject anything at or before frozen/current reference.
    df = df[
        df["timestamp"] > reference
    ].copy()

    df = (
        df.sort_values("timestamp")
        .reset_index(drop=True)
    )

    if len(df) == 0:
        print()
        print("No new closed bars to process.")
        print("STATUS: WAITING_FOR_MARKET_DATA")

        return {
            "passed": True,
            "reason": "NO_NEW_BARS",
            "processed": 0,
        }

    # Market continuity check
    gaps = check_forward_continuity(
        df,
        reference
    )

    if gaps:
        print()
        print("STOP - MARKET DATA GAP DETECTED")
        print()
        print(
            "Do not skip forward in time because the Frozen "
            "features require continuous canonical history."
        )

        display(pd.DataFrame(gaps).head(20))

        return {
            "passed": False,
            "reason": "MARKET_DATA_GAP",
            "gaps": gaps,
        }

    print("Reference timestamp :", reference)
    print("New bars            :", len(df))
    print("First new bar        :", df["timestamp"].iloc[0])
    print("Last new bar         :", df["timestamp"].iloc[-1])

    engine = globals()["run_forward_paper_once"]

    audit_records = []
    results = []
    failures = 0

    print()
    print("=" * 90)
    print("SEQUENTIAL PAPER EXECUTION")
    print("=" * 90)

    for i, row in df.iterrows():

        ts = row["timestamp"]

        one_bar = pd.DataFrame(
            [{
                "timestamp": ts,
                "open": float(row["open"]),
                "high": float(row["high"]),
                "low": float(row["low"]),
                "close": float(row["close"]),
            }]
        )

        try:
            result = engine(one_bar)

            results.append(
                {
                    "timestamp": ts,
                    "result": result,
                }
            )

            processed_times.add(ts)

            action = None
            reason = None

            if isinstance(result, dict):
                action = result.get("action")
                reason = result.get("reason")

            audit_records.append(
                {
                    "timestamp": ts,
                    "status": "ENGINE_OK",
                    "action": action,
                    "reason": reason,
                    "error": "",
                }
            )

            if (
                (i + 1) % 100 == 0
                or i == len(df) - 1
            ):
                print(
                    f"Processed {i + 1:,} / {len(df):,}"
                )

        except Exception as exc:

            failures += 1

            audit_records.append(
                {
                    "timestamp": ts,
                    "status": "ENGINE_ERROR",
                    "action": "",
                    "reason": "",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )

            print()
            print("ENGINE STOP")
            print("Timestamp:", ts)
            print("Error:", type(exc).__name__, str(exc))
            print()
            print(
                "No later bars will be processed because "
                "sequential state must remain intact."
            )

            break

    append_gate_audit(audit_records)

    processed_ok = sum(
        r["status"] == "ENGINE_OK"
        for r in audit_records
    )

    print()
    print("=" * 90)
    print("FORWARD INGESTION RESULT")
    print("=" * 90)
    print("Requested bars :", len(df))
    print("Processed OK   :", processed_ok)
    print("Failures       :", failures)

    if failures == 0 and processed_ok == len(df):

        print()
        print("STATUS: PASS")
        print(
            "All supplied REAL closed 15m bars were processed "
            "sequentially."
        )

        return {
            "passed": True,
            "reason": "FORWARD_BARS_PROCESSED",
            "processed": processed_ok,
            "results": results,
        }

    print()
    print("STATUS: STOP")
    print(
        "Forward Paper execution stopped at the first engine error."
    )

    return {
        "passed": False,
        "reason": "ENGINE_ERROR",
        "processed": processed_ok,
        "failures": failures,
        "results": results,
    }


# ============================================================
# 11. LOAD REAL INCOMING CSV
# ============================================================

def run_forward_ingestion():
    """
    Main command.

    Reads:
        incoming_usdjpy_15m.csv

    Then validates and sends new real bars to the
    Frozen Forward Paper Engine.
    """

    if not incoming_path.exists():
        print("Incoming CSV does not exist:")
        print(incoming_path)

        return {
            "passed": False,
            "reason": "INCOMING_FILE_NOT_FOUND",
        }

    try:
        incoming = pd.read_csv(incoming_path)
    except Exception as exc:
        print(
            "Could not read incoming CSV:",
            type(exc).__name__,
            str(exc)
        )

        return {
            "passed": False,
            "reason": "CSV_READ_ERROR",
        }

    if len(incoming) == 0:
        print()
        print("=" * 90)
        print("WAITING FOR REAL MARKET DATA")
        print("=" * 90)
        print("Incoming CSV:", incoming_path)
        print()
        print("Required columns:")
        print(REQUIRED_COLS)
        print()
        print(
            "No prices were invented. "
            "Forward Paper Engine remains idle."
        )

        return {
            "passed": True,
            "reason": "WAITING_FOR_MARKET_DATA",
            "processed": 0,
        }

    return run_real_forward_bars(incoming)


# ============================================================
# 12. FINAL SYSTEM CHECK
# ============================================================

if (
    len(champion_candidates) > 0
    and engine_exists
    and canonical_df is not None
):

    final_checks = {
        "champion_found": champion_dir.exists(),
        "paper_directory": paper_dir.exists(),
        "incoming_template": incoming_path.exists(),
        "paper_engine_loaded": engine_exists,
        "canonical_history_loaded": canonical_df is not None,
        "real_orders_disabled": True,
    }

    FORWARD_DATA_GATE_READY = all(
        final_checks.values()
    )

    print()
    print("=" * 90)
    print("FINAL DATA GATE CHECK")
    print("=" * 90)

    for k, v in final_checks.items():
        print(f"{k}: {v}")

    print()
    print(
        "FORWARD_DATA_GATE_READY:",
        FORWARD_DATA_GATE_READY
    )

    if FORWARD_DATA_GATE_READY:
        print()
        print("STATUS: PASS")
        print(
            "Real USDJPY 15m ingestion gate is ready."
        )
        print(
            "No live broker orders are possible from this cell."
        )

        print()
        print("=" * 90)
        print("RUN CURRENT INCOMING FILE")
        print("=" * 90)

        FORWARD_INGESTION_RESULT = run_forward_ingestion()

    else:
        FORWARD_INGESTION_RESULT = {
            "passed": False,
            "reason": "DATA_GATE_NOT_READY",
        }

else:
    FORWARD_INGESTION_RESULT = {
        "passed": False,
        "reason": "PRECONDITION_FAILED",
    }


print()
print("=" * 90)
print("CELL COMPLETE")
print("=" * 90)

print(
    "FORWARD_DATA_GATE_READY:",
    globals().get(
        "FORWARD_DATA_GATE_READY",
        False
    )
)

print(
    "Result:",
    globals().get(
        "FORWARD_INGESTION_RESULT",
        {}
    )
)


## 元セルindex 88
構文状態：valid


In [ ]:
# ============================================================
# USDJPY 15m MARKET DATA ADAPTER
# Real closed 15m bars -> validation -> Forward Paper Engine
# PAPER ONLY / NO BROKER ORDER
# ============================================================

from pathlib import Path
import json
from datetime import datetime

import pandas as pd
import numpy as np


# ============================================================
# 0. FROZEN CONTRACT
# ============================================================

CHAMPION_VERSION = "champion_v1_base_plus_regime_20260909"

CHAMPION_DIR = (
    Path("production_champion")
    / CHAMPION_VERSION
)

PAPER_DIR = (
    CHAMPION_DIR
    / "runtime"
    / "paper_forward"
)

INCOMING_CSV = (
    PAPER_DIR
    / "incoming_usdjpy_15m.csv"
)

NORMALIZED_CSV = (
    PAPER_DIR
    / "market_data_adapter_normalized.csv"
)

STATE_JSON = (
    PAPER_DIR
    / "market_data_adapter_state.json"
)

AUDIT_CSV = (
    PAPER_DIR
    / "market_data_adapter_audit.csv"
)

CONTRACT_JSON = (
    PAPER_DIR
    / "market_data_adapter_contract.json"
)


# Frozen Champion時点の最後の15分足
FROZEN_LATEST = pd.Timestamp(
    "2026-09-01 00:00:00",
    tz="UTC"
)


REQUIRED = [
    "timestamp",
    "open",
    "high",
    "low",
    "close",
]


PAPER_ONLY = True

REAL_ORDERS_ENABLED = False



# ============================================================
# 1. HARD PRECHECK
# ============================================================

if REAL_ORDERS_ENABLED:

    raise RuntimeError(
        "SAFETY STOP: "
        "REAL_ORDERS_ENABLED must be False."
    )


if not CHAMPION_DIR.exists():

    raise FileNotFoundError(
        "Frozen Champion directory not found:\n"
        f"{CHAMPION_DIR}"
    )


PAPER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


if (
    "run_forward_paper_once" not in globals()
    or not callable(run_forward_paper_once)
):

    raise RuntimeError(
        "run_forward_paper_once() is not loaded.\n"
        "Run the Forward Paper Trading System / "
        "Data Gate cells first."
    )


# incoming CSVがまだ無ければ
# 空テンプレートだけ作成
if not INCOMING_CSV.exists():

    pd.DataFrame(
        columns=REQUIRED
    ).to_csv(
        INCOMING_CSV,
        index=False,
    )



# ============================================================
# 2. SMALL HELPERS
# ============================================================

def _utc_ts(x):

    ts = pd.Timestamp(x)

    if ts.tzinfo is None:

        return ts.tz_localize("UTC")

    return ts.tz_convert("UTC")



def _read_state():

    if not STATE_JSON.exists():

        return {}

    try:

        with open(
            STATE_JSON,
            "r",
            encoding="utf-8",
        ) as f:

            return json.load(f)

    except Exception:

        return {}



def _write_state(obj):

    tmp = STATE_JSON.with_suffix(
        ".json.tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    tmp.replace(
        STATE_JSON
    )



def _append_audit(row):

    out = pd.DataFrame(
        [row]
    )

    if AUDIT_CSV.exists():

        out.to_csv(
            AUDIT_CSV,
            mode="a",
            header=False,
            index=False,
        )

    else:

        out.to_csv(
            AUDIT_CSV,
            index=False,
        )



def _latest_closed_15m_start(
    now=None
):

    """
    timestamp は
    15分足の開始時刻として扱う。

    例:
    現在 UTC 12:17
    ↓
    12:00開始の15分足が
    最新の完全確定足。
    """

    if now is None:

        now = pd.Timestamp.now(
            tz="UTC"
        )

    else:

        now = _utc_ts(
            now
        )

    return (
        now.floor("15min")
        - pd.Timedelta(
            minutes=15
        )
    )



# ============================================================
# 3. MARKET DATA NORMALIZATION
# ============================================================

def _normalize_real_bars(
    df
):

    if not isinstance(
        df,
        pd.DataFrame,
    ):

        raise TypeError(
            "Input must be a pandas DataFrame."
        )


    x = df.copy()


    # -------------------------
    # column names
    # -------------------------

    x.columns = [
        str(c)
        .strip()
        .lower()
        for c in x.columns
    ]


    # DatetimeIndexも許可
    if (
        "timestamp" not in x.columns
        and isinstance(
            x.index,
            pd.DatetimeIndex,
        )
    ):

        x = x.reset_index()

        x = x.rename(
            columns={
                x.columns[0]:
                "timestamp"
            }
        )


    # -------------------------
    # required columns
    # -------------------------

    missing = [
        c
        for c in REQUIRED
        if c not in x.columns
    ]


    if missing:

        raise ValueError(
            "Missing required columns: "
            f"{missing}"
        )


    x = x[
        REQUIRED
    ].copy()


    # -------------------------
    # timestamp -> UTC
    # -------------------------

    x["timestamp"] = pd.to_datetime(
        x["timestamp"],
        utc=True,
        errors="coerce",
    )


    # -------------------------
    # OHLC -> numeric
    # -------------------------

    for c in [
        "open",
        "high",
        "low",
        "close",
    ]:

        x[c] = pd.to_numeric(
            x[c],
            errors="coerce",
        )


    # -------------------------
    # timestamp validity
    # -------------------------

    if x["timestamp"].isna().any():

        raise ValueError(
            "Invalid timestamp rows: "
            f"{int(x['timestamp'].isna().sum())}"
        )


    # -------------------------
    # OHLC validity
    # -------------------------

    bad_price_row = (
        x[
            [
                "open",
                "high",
                "low",
                "close",
            ]
        ]
        .isna()
        .any(axis=1)
    )


    if bad_price_row.any():

        raise ValueError(
            "Invalid OHLC rows: "
            f"{int(bad_price_row.sum())}"
        )


    if (
        x[
            [
                "open",
                "high",
                "low",
                "close",
            ]
        ]
        <= 0
    ).any().any():

        raise ValueError(
            "Non-positive OHLC price detected."
        )


    # -------------------------
    # OHLC relationship
    # -------------------------

    bad_ohlc = (

        x["high"]

        < x[
            [
                "open",
                "close",
                "low",
            ]
        ].max(axis=1)

    ) | (

        x["low"]

        > x[
            [
                "open",
                "close",
                "high",
            ]
        ].min(axis=1)

    )


    if bad_ohlc.any():

        raise ValueError(
            "OHLC relationship invalid: "
            f"{int(bad_ohlc.sum())} row(s)"
        )


    # ========================================================
    # EXACT 15-MINUTE GRID
    # ========================================================

    grid_ok = (

        (
            x["timestamp"].dt.minute
            % 15
            == 0
        )

        & (
            x["timestamp"].dt.second
            == 0
        )

        & (
            x["timestamp"].dt.microsecond
            == 0
        )

    )


    if not grid_ok.all():

        examples = (

            x.loc[
                ~grid_ok,
                "timestamp",
            ]

            .astype(str)

            .head(5)

            .tolist()
        )


        raise ValueError(

            "Off-grid timestamp detected.\n"

            "timestamp must be UTC BAR START "
            "time on the exact 15m grid.\n"

            f"Examples: {examples}"
        )


    # ========================================================
    # SORT
    # ========================================================

    x = (

        x.sort_values(
            "timestamp"
        )

        .reset_index(
            drop=True
        )

    )


    # ========================================================
    # DUPLICATE CHECK
    # ========================================================

    if x[
        "timestamp"
    ].duplicated(
        keep=False
    ).any():

        conflict_times = []


        for ts, g in x.groupby(
            "timestamp",
            sort=False,
        ):

            if len(g) <= 1:

                continue


            different = (

                g[
                    [
                        "open",
                        "high",
                        "low",
                        "close",
                    ]
                ]

                .nunique(
                    dropna=False
                )

                .gt(1)

                .any()

            )


            if different:

                conflict_times.append(
                    str(ts)
                )


        # 同じtimestampで
        # OHLCが違えば停止
        if conflict_times:

            raise ValueError(

                "Conflicting duplicate "
                "market bars detected.\n"

                f"Examples: "
                f"{conflict_times[:5]}"

            )


        # 完全に同じ重複なら
        # 1本にまとめる
        x = (

            x.drop_duplicates(
                subset=[
                    "timestamp"
                ],
                keep="last",
            )

            .reset_index(
                drop=True
            )

        )


    return x



# ============================================================
# 4. ENGINE RESULT NORMALIZER
# ============================================================

def _engine_result_dict(
    result
):

    if isinstance(
        result,
        dict,
    ):

        return result


    if result is None:

        return {
            "result": None
        }


    return {
        "result": repr(result)
    }



# ============================================================
# 5. SAVE MARKET DATA ADAPTER CONTRACT
# ============================================================

adapter_contract = {

    "version":
        "market_data_adapter_v1",

    "champion":
        "BASE_PLUS_REGIME",

    "champion_version":
        CHAMPION_VERSION,

    "pair":
        "USDJPY",

    "timeframe":
        "15m",

    "timestamp_semantics":
        "UTC_BAR_START",

    "required_columns":
        REQUIRED,

    "frozen_latest":
        str(FROZEN_LATEST),

    "paper_only":
        True,

    "real_orders_enabled":
        False,

    "processing":
        "ONE_CLOSED_BAR_AT_A_TIME",

}


with open(
    CONTRACT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        adapter_contract,
        f,
        ensure_ascii=False,
        indent=2,
    )



# ============================================================
# 6. MAIN MARKET DATA ADAPTER
# ============================================================

def run_market_data_adapter_once(
    market_df=None,
    now=None,
    verbose=True,
):

    """
    market_df=None:

        incoming_usdjpy_15m.csv
        を読み込む。


    market_df=DataFrame:

        API等から取得した
        DataFrameを直接渡す。


    Required columns:

        timestamp
        open
        high
        low
        close


    timestamp:

        UTC BAR START
    """


    # ========================================================
    # LOAD MARKET DATA
    # ========================================================

    if market_df is None:


        if not INCOMING_CSV.exists():

            raw = pd.DataFrame(
                columns=REQUIRED
            )


        else:

            try:

                raw = pd.read_csv(
                    INCOMING_CSV
                )

            except pd.errors.EmptyDataError:

                raw = pd.DataFrame(
                    columns=REQUIRED
                )


        source = str(
            INCOMING_CSV
        )


    else:


        raw = market_df.copy()

        source = (
            "PROVIDER_DATAFRAME"
        )


    # ========================================================
    # NO DATA
    # ========================================================

    if raw.empty:


        result = {

            "passed":
                True,

            "reason":
                "WAITING_FOR_MARKET_DATA",

            "processed":
                0,

        }


        if verbose:

            print(
                "=" * 88
            )

            print(
                "MARKET DATA ADAPTER"
            )

            print(
                "=" * 88
            )

            print(
                "STATUS: "
                "WAITING_FOR_MARKET_DATA"
            )

            print(
                "No real USDJPY "
                "15m bars supplied."
            )

            print(
                "No fake prices "
                "were generated."
            )


        return result



    # ========================================================
    # NORMALIZE
    # ========================================================

    bars = _normalize_real_bars(
        raw
    )


    bars.to_csv(
        NORMALIZED_CSV,
        index=False,
    )



    # ========================================================
    # LOAD STATE
    # ========================================================

    state = _read_state()


    if state.get(
        "last_processed_timestamp"
    ):


        last_processed = _utc_ts(

            state[
                "last_processed_timestamp"
            ]

        )


    else:


        last_processed = (
            FROZEN_LATEST
        )



    cutoff = max(

        FROZEN_LATEST,

        last_processed,

    )



    # ========================================================
    # CLOSED BAR CUT-OFF
    # ========================================================

    latest_closed = (
        _latest_closed_15m_start(
            now
        )
    )


    # 未確定足は完全無視
    closed = bars[

        bars["timestamp"]
        <= latest_closed

    ].copy()



    # ========================================================
    # ONLY NEW BARS
    # ========================================================

    new_bars = (

        closed[

            closed["timestamp"]
            > cutoff

        ]

        .sort_values(
            "timestamp"
        )

        .reset_index(
            drop=True
        )

    )



    # ========================================================
    # NOTHING NEW
    # ========================================================

    if new_bars.empty:


        result = {

            "passed":
                True,

            "reason":
                "WAITING_FOR_NEW_CLOSED_15M_BARS",

            "processed":
                0,

            "last_processed":
                str(cutoff),

            "latest_closed_bar_start":
                str(latest_closed),

        }


        if verbose:


            print(
                "=" * 88
            )

            print(
                "MARKET DATA ADAPTER"
            )

            print(
                "=" * 88
            )

            print(
                "STATUS: "
                "WAITING_FOR_NEW_CLOSED_15M_BARS"
            )

            print(
                "Supplied rows:",
                len(bars),
            )

            print(
                "Last processed:",
                cutoff,
            )

            print(
                "Latest closed "
                "15m bar start:",
                latest_closed,
            )


        return result



    # ========================================================
    # SEQUENTIAL LIVE-STYLE PROCESSING
    # ========================================================

    engine_results = []

    processed = 0



    for _, row in new_bars.iterrows():


        ts = _utc_ts(
            row["timestamp"]
        )


        one_bar = pd.DataFrame(

            [{

                "timestamp":
                    ts,

                "open":
                    float(
                        row["open"]
                    ),

                "high":
                    float(
                        row["high"]
                    ),

                "low":
                    float(
                        row["low"]
                    ),

                "close":
                    float(
                        row["close"]
                    ),

            }]

        )



        try:


            # =================================================
            # FORWARD PAPER ENGINE
            # =================================================

            engine_result = (
                run_forward_paper_once(
                    one_bar
                )
            )


            engine_dict = (
                _engine_result_dict(
                    engine_result
                )
            )



            # =================================================
            # EXPLICIT FAILURE -> STOP
            # =================================================

            if (
                engine_dict.get(
                    "passed",
                    True,
                )
                is False
            ):


                _append_audit({

                    "audit_time_utc":
                        str(
                            pd.Timestamp.now(
                                tz="UTC"
                            )
                        ),

                    "bar_timestamp":
                        str(ts),

                    "source":
                        source,

                    "status":
                        "ENGINE_REJECTED",

                    "engine_result":
                        json.dumps(
                            engine_dict,
                            ensure_ascii=False,
                            default=str,
                        ),

                })


                raise RuntimeError(

                    "Forward Paper Engine "
                    "rejected bar "
                    f"{ts}: "
                    f"{engine_dict}"

                )



            # =================================================
            # SUCCESS
            # =================================================

            processed += 1


            # 状態は
            # Engine成功後だけ更新
            state = {

                "champion_version":
                    CHAMPION_VERSION,

                "last_processed_timestamp":
                    str(ts),

                "updated_at_utc":
                    str(
                        pd.Timestamp.now(
                            tz="UTC"
                        )
                    ),

                "paper_only":
                    True,

            }


            _write_state(
                state
            )



            _append_audit({

                "audit_time_utc":
                    str(
                        pd.Timestamp.now(
                            tz="UTC"
                        )
                    ),

                "bar_timestamp":
                    str(ts),

                "source":
                    source,

                "status":
                    "PROCESSED",

                "engine_result":
                    json.dumps(
                        engine_dict,
                        ensure_ascii=False,
                        default=str,
                    ),

            })



            engine_results.append({

                "timestamp":
                    str(ts),

                "result":
                    engine_dict,

            })



        except Exception as e:


            _append_audit({

                "audit_time_utc":
                    str(
                        pd.Timestamp.now(
                            tz="UTC"
                        )
                    ),

                "bar_timestamp":
                    str(ts),

                "source":
                    source,

                "status":
                    "ERROR",

                "engine_result":
                    repr(e),

            })


            print(
                "=" * 88
            )

            print(
                "ADAPTER STOPPED"
            )

            print(
                "=" * 88
            )

            print(
                "Failed bar:",
                ts,
            )

            print(
                "Error:",
                repr(e),
            )

            print(
                "State was NOT advanced "
                "beyond the failed bar."
            )


            raise



    # ========================================================
    # FINAL RESULT
    # ========================================================

    result = {

        "passed":
            True,

        "reason":
            "PROCESSED_NEW_CLOSED_15M_BARS",

        "processed":
            processed,

        "last_processed":
            state[
                "last_processed_timestamp"
            ],

        "engine_results":
            engine_results,

    }



    if verbose:


        print(
            "=" * 88
        )

        print(
            "MARKET DATA ADAPTER COMPLETE"
        )

        print(
            "=" * 88
        )


        print(
            "Champion:",
            CHAMPION_VERSION,
        )


        print(
            "Frozen latest:",
            FROZEN_LATEST,
        )


        print(
            "Processed:",
            processed,
        )


        print(
            "Last processed:",
            state[
                "last_processed_timestamp"
            ],
        )


        print(
            "Paper only:",
            PAPER_ONLY,
        )


        print(
            "Real orders enabled:",
            REAL_ORDERS_ENABLED,
        )


        print()


        print(
            "NEXT:"
        )


        print(
            "Connect a real USDJPY "
            "15m market-data provider."
        )


        print(
            "Then pass the provider "
            "DataFrame into:"
        )


        print(
            "run_market_data_adapter_once("
            "provider_dataframe)"
        )



    return result



# ============================================================
# 7. BUILD / SMOKE TEST
# ============================================================

print(
    "=" * 88
)

print(
    "USDJPY 15m MARKET DATA ADAPTER BUILD"
)

print(
    "=" * 88
)


print(
    "Champion:",
    CHAMPION_VERSION,
)


print(
    "Champion directory:",
    CHAMPION_DIR,
)


print(
    "Paper directory:",
    PAPER_DIR,
)


print(
    "Incoming CSV:",
    INCOMING_CSV,
)


print(
    "Frozen latest:",
    FROZEN_LATEST,
)


print(
    "run_forward_paper_once loaded:",
    callable(
        run_forward_paper_once
    ),
)


print(
    "Paper only:",
    PAPER_ONLY,
)


print(
    "Real orders enabled:",
    REAL_ORDERS_ENABLED,
)


print()



MARKET_DATA_ADAPTER_RESULT = (
    run_market_data_adapter_once()
)



print()

print(
    "=" * 88
)

print(
    "CELL COMPLETE"
)

print(
    "=" * 88
)


print(
    "Result:",
    MARKET_DATA_ADAPTER_RESULT,
)


## 元セルindex 89
構文状態：valid


In [ ]:
# ============================================================
# USDJPY 15m REAL MARKET DATA CONNECTOR
# -> FROZEN FORWARD PAPER ENGINE
#
# 目的:
#   1. Twelve Data から本物の USDJPY 15分足を取得
#   2. UTC / OHLC / 15分グリッド / 確定足を検査
#   3. 初回は ARM のみ（過去足をForward成績に混ぜない）
#   4. 2回目以降、新しく確定した足だけPaper Engineへ送る
#
# IMPORTANT:
#   - Real order = DISABLED
#   - Historical backfill = DISABLED
#   - Frozen Champion = MODIFYしない
# ============================================================

import os
import json
import getpass
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

import pandas as pd
import numpy as np


# ============================================================
# 0. CONFIG
# ============================================================

SYMBOL = "USD/JPY"
INTERVAL = "15min"

# 直近100本だけ取得。
# Forward運用には十分で、API消費も抑える。
OUTPUTSIZE = 100

# API側で形成中の足を誤って取得しないための安全マージン。
CLOSE_SAFETY_SECONDS = 60

CHAMPION_VERSION = "champion_v1_base_plus_regime_20260909"

CHAMPION_DIR = (
    Path("production_champion")
    / CHAMPION_VERSION
)

PAPER_DIR = (
    CHAMPION_DIR
    / "runtime"
    / "paper_forward"
)

PAPER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

STATE_FILE = (
    PAPER_DIR
    / "market_data_state.json"
)

INCOMING_FILE = (
    PAPER_DIR
    / "incoming_usdjpy_15m.csv"
)

SNAPSHOT_FILE = (
    PAPER_DIR
    / "market_data_source_snapshot.csv"
)

AUDIT_FILE = (
    PAPER_DIR
    / "market_data_connector_audit.csv"
)


# ============================================================
# 1. SMALL UTILITIES
# ============================================================

def section(title):

    print()
    print("=" * 88)
    print(title)
    print("=" * 88)


def utc_now():

    return pd.Timestamp.now(
        tz="UTC"
    )


def latest_closed_bar_start():

    """
    現在時刻から見て、
    完全に確定している最新15分足の開始時刻を返す。

    例:
        現在 13:31 UTC

        13:15足:
            13:15 -> 13:30

        これは確定済みなので使用可能。
    """

    now = utc_now()

    safe_now = (
        now
        - pd.Timedelta(
            seconds=CLOSE_SAFETY_SECONDS
        )
    )

    current_bucket = (
        safe_now.floor("15min")
    )

    latest_closed = (
        current_bucket
        - pd.Timedelta(minutes=15)
    )

    return latest_closed


def save_audit(
    status,
    detail="",
    rows=0,
    latest_bar=None
):

    new_row = pd.DataFrame(
        [
            {
                "timestamp_utc":
                    utc_now().isoformat(),

                "status":
                    str(status),

                "detail":
                    str(detail),

                "rows":
                    int(rows),

                "latest_bar":
                    ""
                    if latest_bar is None
                    else pd.Timestamp(
                        latest_bar
                    ).isoformat()
            }
        ]
    )

    if AUDIT_FILE.exists():

        try:

            old = pd.read_csv(
                AUDIT_FILE
            )

            new_row = pd.concat(
                [
                    old,
                    new_row
                ],
                ignore_index=True
            )

        except Exception:

            pass

    new_row.to_csv(
        AUDIT_FILE,
        index=False
    )


# ============================================================
# 2. LOCATE EXISTING PAPER ENGINE
# ============================================================

def find_paper_engine():

    """
    これまで作った関数を自動検出。

    優先:
        run_market_data_adapter_once

    fallback:
        run_forward_paper_once

    モデルのfallbackではない。
    既存Paper Engineの入口名の違いだけを吸収する。
    """

    candidates = [

        "run_market_data_adapter_once",

        "run_forward_paper_once"

    ]

    for name in candidates:

        obj = globals().get(name)

        if callable(obj):

            return name, obj

    return None, None


# ============================================================
# 3. API KEY
# ============================================================

def get_api_key():

    """
    環境変数があれば使用。

    なければJupyter上で入力。

    getpassなのでAPIキーは画面に表示されない。
    """

    key = os.getenv(
        "TWELVE_DATA_API_KEY",
        ""
    ).strip()

    if key:

        return key

    try:

        key = getpass.getpass(

            "Twelve Data API keyを入力"
            "（入力内容は表示されません）: "

        ).strip()

    except Exception:

        key = ""

    return key


# ============================================================
# 4. FETCH REAL USDJPY 15m DATA
# ============================================================

def fetch_real_usdjpy(api_key):

    """
    Twelve Data time_series APIから

        USD/JPY
        15min
        UTC
        OHLC

    を取得する。
    """

    base = (
        "https://"
        + "api.twelvedata.com"
    )

    endpoint = (
        base
        + "/time_series"
    )

    params = {

        "symbol":
            SYMBOL,

        "interval":
            INTERVAL,

        "outputsize":
            OUTPUTSIZE,

        "timezone":
            "UTC",

        "order":
            "asc",

        "format":
            "JSON",

        "apikey":
            api_key
    }

    url = (
        endpoint
        + "?"
        + urlencode(params)
    )

    request = Request(

        url,

        headers={
            "User-Agent":
                "FX-Forward-Paper/1.0"
        }
    )

    try:

        with urlopen(
            request,
            timeout=25
        ) as response:

            text = (
                response
                .read()
                .decode("utf-8")
            )

    except HTTPError as e:

        return (
            None,
            f"HTTP_ERROR_{e.code}"
        )

    except URLError as e:

        return (
            None,
            f"NETWORK_ERROR: {e}"
        )

    except Exception as e:

        return (
            None,
            f"REQUEST_ERROR: "
            f"{type(e).__name__}: {e}"
        )

    try:

        payload = json.loads(
            text
        )

    except Exception as e:

        return (
            None,
            f"JSON_ERROR: "
            f"{type(e).__name__}: {e}"
        )

    if (
        isinstance(payload, dict)
        and
        payload.get("status") == "error"
    ):

        return (
            None,
            "API_ERROR: "
            + str(
                payload.get(
                    "message",
                    payload
                )
            )
        )

    values = (
        payload.get("values")
        if isinstance(payload, dict)
        else None
    )

    if not isinstance(
        values,
        list
    ):

        return (
            None,
            "NO_VALUES_RETURNED"
        )

    if len(values) == 0:

        return (
            None,
            "EMPTY_VALUES"
        )

    df = pd.DataFrame(
        values
    )

    required = [

        "datetime",
        "open",
        "high",
        "low",
        "close"

    ]

    missing = [

        c
        for c in required
        if c not in df.columns

    ]

    if missing:

        return (
            None,
            f"MISSING_COLUMNS: {missing}"
        )

    df = df[
        required
    ].copy()

    # UTC timestamp
    df["timestamp"] = pd.to_datetime(

        df["datetime"],

        utc=True,

        errors="coerce"
    )

    # OHLC numeric化
    for col in [

        "open",
        "high",
        "low",
        "close"

    ]:

        df[col] = pd.to_numeric(

            df[col],

            errors="coerce"
        )

    df = (
        df
        .drop(columns=["datetime"])
        .dropna(
            subset=[
                "timestamp",
                "open",
                "high",
                "low",
                "close"
            ]
        )
        .drop_duplicates(
            subset=["timestamp"],
            keep="last"
        )
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    return df, None


# ============================================================
# 5. MARKET DATA VALIDATION
# ============================================================

def validate_bars(df):

    problems = []

    if (
        df is None
        or
        len(df) == 0
    ):

        return (
            False,
            ["EMPTY_DATA"]
        )

    timestamps = pd.DatetimeIndex(
        df["timestamp"]
    )

    # --------
    # 15m grid
    # --------

    exact_grid = (

        (timestamps.minute % 15 == 0)

        &

        (timestamps.second == 0)

        &

        (timestamps.microsecond == 0)

    )

    if not bool(
        np.all(exact_grid)
    ):

        problems.append(
            "OFF_15M_GRID"
        )

    # --------
    # duplicates
    # --------

    duplicate_count = int(

        df["timestamp"]
        .duplicated()
        .sum()

    )

    if duplicate_count != 0:

        problems.append(

            f"DUPLICATES="
            f"{duplicate_count}"

        )

    # --------
    # ordering
    # --------

    if not df[
        "timestamp"
    ].is_monotonic_increasing:

        problems.append(
            "NOT_ASCENDING"
        )

    # --------
    # positive price
    # --------

    prices = df[
        [
            "open",
            "high",
            "low",
            "close"
        ]
    ]

    if not bool(
        (prices > 0)
        .all()
        .all()
    ):

        problems.append(
            "NON_POSITIVE_PRICE"
        )

    # --------
    # OHLC consistency
    # --------

    o = df["open"]

    h = df["high"]

    l = df["low"]

    c = df["close"]

    valid_ohlc = (

        (h >= o)

        &

        (h >= c)

        &

        (l <= o)

        &

        (l <= c)

        &

        (h >= l)

    )

    if not bool(
        valid_ohlc.all()
    ):

        problems.append(
            "INVALID_OHLC"
        )

    return (
        len(problems) == 0,
        problems
    )


# ============================================================
# 6. MAIN
# ============================================================

section(
    "USDJPY REAL MARKET DATA -> FORWARD PAPER"
)

print(
    "Champion:",
    CHAMPION_VERSION
)

print(
    "Symbol:",
    SYMBOL
)

print(
    "Interval:",
    INTERVAL
)

print(
    "Paper only:",
    True
)

print(
    "Real broker orders:",
    False
)

print(
    "Historical backfill:",
    False
)


# ------------------------------------------------------------
# Paper Engine確認
# ------------------------------------------------------------

engine_name, engine = (
    find_paper_engine()
)

print(
    "Detected Paper Engine:",
    engine_name
)


if engine is None:

    MARKET_CONNECTOR_RESULT = {

        "passed":
            False,

        "reason":
            "PAPER_ENGINE_NOT_FOUND",

        "processed":
            0
    }

    save_audit(
        "STOP",
        "PAPER_ENGINE_NOT_FOUND"
    )

    print()
    print(
        "STOP:"
        " Paper Engine関数が"
        "見つかりません。"
    )


else:

    # --------------------------------------------------------
    # API KEY
    # --------------------------------------------------------

    api_key = (
        get_api_key()
    )

    if not api_key:

        MARKET_CONNECTOR_RESULT = {

            "passed":
                False,

            "reason":
                "API_KEY_MISSING",

            "processed":
                0
        }

        save_audit(
            "STOP",
            "API_KEY_MISSING"
        )

        print()
        print(
            "STOP:"
            " API keyがありません。"
        )


    else:

        print(
            "API key:",
            "LOADED (hidden)"
        )

        # ----------------------------------------------------
        # FETCH
        # ----------------------------------------------------

        bars, error = (
            fetch_real_usdjpy(
                api_key
            )
        )

        if error is not None:

            MARKET_CONNECTOR_RESULT = {

                "passed":
                    False,

                "reason":
                    error,

                "processed":
                    0
            }

            save_audit(
                "STOP",
                error
            )

            print()
            print(
                "STOP:"
            )

            print(
                error
            )


        else:

            # ------------------------------------------------
            # VALIDATE
            # ------------------------------------------------

            data_ok, problems = (
                validate_bars(
                    bars
                )
            )

            if not data_ok:

                MARKET_CONNECTOR_RESULT = {

                    "passed":
                        False,

                    "reason":
                        "DATA_VALIDATION_FAILED",

                    "problems":
                        problems,

                    "processed":
                        0
                }

                save_audit(

                    "STOP",

                    ";".join(
                        problems
                    )
                )

                print()
                print(
                    "STOP:"
                    " Market data validation failed."
                )

                print(
                    problems
                )


            else:

                # --------------------------------------------
                # CLOSED BARS ONLY
                # --------------------------------------------

                max_closed = (
                    latest_closed_bar_start()
                )

                closed = (
                    bars[
                        bars["timestamp"]
                        <= max_closed
                    ]
                    .copy()
                    .sort_values(
                        "timestamp"
                    )
                    .reset_index(
                        drop=True
                    )
                )

                if len(closed) == 0:

                    MARKET_CONNECTOR_RESULT = {

                        "passed":
                            True,

                        "reason":
                            "WAITING_FOR_CLOSED_BAR",

                        "processed":
                            0
                    }

                    save_audit(
                        "WAIT",
                        "NO_CLOSED_BAR"
                    )

                    print()
                    print(
                        "WAITING FOR "
                        "CLOSED 15m BAR"
                    )


                else:

                    closed.to_csv(

                        SNAPSHOT_FILE,

                        index=False
                    )

                    latest_market_bar = (
                        pd.Timestamp(
                            closed[
                                "timestamp"
                            ].iloc[-1]
                        )
                    )

                    section(
                        "REAL MARKET DATA CHECK"
                    )

                    print(
                        "Rows fetched:",
                        len(bars)
                    )

                    print(
                        "Closed rows:",
                        len(closed)
                    )

                    print(
                        "First:",
                        closed[
                            "timestamp"
                        ].iloc[0]
                    )

                    print(
                        "Latest:",
                        latest_market_bar
                    )

                    print(
                        "15m grid:",
                        True
                    )

                    print(
                        "OHLC integrity:",
                        True
                    )

                    # ========================================
                    # FIRST RUN
                    # ========================================

                    if not STATE_FILE.exists():

                        state = {

                            "version":
                                1,

                            "champion":
                                CHAMPION_VERSION,

                            "symbol":
                                SYMBOL,

                            "interval":
                                INTERVAL,

                            "mode":
                                "FORWARD_ONLY_NO_BACKFILL",

                            "activated_at_utc":
                                utc_now().isoformat(),

                            "activation_cutoff_bar":
                                latest_market_bar.isoformat(),

                            "last_processed_market_bar":
                                latest_market_bar.isoformat()
                        }

                        STATE_FILE.write_text(

                            json.dumps(
                                state,
                                indent=2
                            ),

                            encoding="utf-8"
                        )

                        save_audit(

                            "ARMED",

                            "FIRST_RUN_NO_BACKFILL",

                            latest_bar=
                                latest_market_bar
                        )

                        section(
                            "FORWARD PAPER ARMED"
                        )

                        print(
                            "Connection:",
                            "PASS"
                        )

                        print(
                            "Historical bars sent:",
                            0
                        )

                        print(
                            "Activation cutoff:",
                            latest_market_bar
                        )

                        print(
                            "State persistence:",
                            True
                        )

                        print()
                        print(
                            "次に確定する15分足から"
                            "Forward Paperを開始します。"
                        )

                        MARKET_CONNECTOR_RESULT = {

                            "passed":
                                True,

                            "reason":
                                "ARMED_WAITING_FOR_NEXT_BAR",

                            "processed":
                                0,

                            "activation_cutoff":
                                latest_market_bar
                        }


                    # ========================================
                    # SECOND+ RUNS
                    # ========================================

                    else:

                        try:

                            state = json.loads(

                                STATE_FILE.read_text(
                                    encoding="utf-8"
                                )
                            )

                            last_processed = (
                                pd.to_datetime(

                                    state[
                                        "last_processed_market_bar"
                                    ],

                                    utc=True
                                )
                            )

                        except Exception as e:

                            MARKET_CONNECTOR_RESULT = {

                                "passed":
                                    False,

                                "reason":
                                    "STATE_FILE_ERROR",

                                "processed":
                                    0
                            }

                            save_audit(

                                "STOP",

                                str(e)
                            )

                            print()
                            print(
                                "STOP:"
                                " State file error"
                            )


                        else:

                            # --------------------------------
                            # NEW BARS ONLY
                            # --------------------------------

                            new_bars = (
                                closed[
                                    closed[
                                        "timestamp"
                                    ]
                                    >
                                    last_processed
                                ]
                                .copy()
                                .sort_values(
                                    "timestamp"
                                )
                                .reset_index(
                                    drop=True
                                )
                            )

                            if len(
                                new_bars
                            ) == 0:

                                MARKET_CONNECTOR_RESULT = {

                                    "passed":
                                        True,

                                    "reason":
                                        "WAITING_FOR_NEW_BAR",

                                    "processed":
                                        0
                                }

                                save_audit(

                                    "WAIT",

                                    "NO_NEW_BAR",

                                    latest_bar=
                                        last_processed
                                )

                                section(
                                    "WAITING FOR NEXT BAR"
                                )

                                print(
                                    "Last processed:",
                                    last_processed
                                )

                                print(
                                    "Latest available:",
                                    latest_market_bar
                                )

                                print(
                                    "Duplicate processing:",
                                    False
                                )


                            else:

                                # ----------------------------
                                # Save exact incoming batch
                                # ----------------------------

                                new_bars.to_csv(

                                    INCOMING_FILE,

                                    index=False
                                )

                                section(
                                    "NEW CLOSED BARS"
                                )

                                print(
                                    "New bars:",
                                    len(new_bars)
                                )

                                print(
                                    "First:",
                                    new_bars[
                                        "timestamp"
                                    ].iloc[0]
                                )

                                print(
                                    "Last:",
                                    new_bars[
                                        "timestamp"
                                    ].iloc[-1]
                                )


                                # ============================
                                # SEND ONE BAR AT A TIME
                                # ============================

                                processed = 0

                                failure = None

                                last_success = (
                                    last_processed
                                )

                                for i in range(
                                    len(new_bars)
                                ):

                                    one_bar = (
                                        new_bars
                                        .iloc[[i]]
                                        .copy()
                                    )

                                    bar_time = (
                                        pd.Timestamp(

                                            one_bar[
                                                "timestamp"
                                            ].iloc[0]

                                        )
                                    )

                                    try:

                                        result = (
                                            engine(
                                                one_bar
                                            )
                                        )

                                    except Exception as e:

                                        failure = (

                                            "PAPER_ENGINE_ERROR "
                                            f"@ {bar_time}: "
                                            f"{type(e).__name__}: "
                                            f"{e}"
                                        )

                                        break


                                    if (
                                        isinstance(
                                            result,
                                            dict
                                        )
                                        and
                                        result.get(
                                            "passed"
                                        )
                                        is False
                                    ):

                                        failure = (

                                            "PAPER_ENGINE_REJECTED "
                                            f"@ {bar_time}: "
                                            f"{result.get('reason')}"
                                        )

                                        break


                                    # ------------------------
                                    # successful bar
                                    # ------------------------

                                    processed += 1

                                    last_success = (
                                        bar_time
                                    )

                                    state[
                                        "last_processed_market_bar"
                                    ] = (
                                        bar_time.isoformat()
                                    )

                                    state[
                                        "last_run_utc"
                                    ] = (
                                        utc_now()
                                        .isoformat()
                                    )

                                    STATE_FILE.write_text(

                                        json.dumps(
                                            state,
                                            indent=2
                                        ),

                                        encoding="utf-8"
                                    )


                                # ============================
                                # FINAL
                                # ============================

                                if failure is not None:

                                    save_audit(

                                        "STOP",

                                        failure,

                                        rows=processed,

                                        latest_bar=
                                            last_success
                                    )

                                    MARKET_CONNECTOR_RESULT = {

                                        "passed":
                                            False,

                                        "reason":
                                            failure,

                                        "processed":
                                            processed
                                    }

                                    print()
                                    print(
                                        "STOP:"
                                    )

                                    print(
                                        failure
                                    )


                                else:

                                    save_audit(

                                        "PASS",

                                        "NEW_BARS_PROCESSED",

                                        rows=processed,

                                        latest_bar=
                                            last_success
                                    )

                                    section(
                                        "FORWARD PAPER PASS"
                                    )

                                    print(
                                        "Processed bars:",
                                        processed
                                    )

                                    print(
                                        "Last processed:",
                                        last_success
                                    )

                                    print(
                                        "Paper only:",
                                        True
                                    )

                                    print(
                                        "Real broker orders:",
                                        False
                                    )

                                    print(
                                        "State persistence:",
                                        True
                                    )

                                    MARKET_CONNECTOR_RESULT = {

                                        "passed":
                                            True,

                                        "reason":
                                            "NEW_BARS_PROCESSED",

                                        "processed":
                                            processed,

                                        "last_processed":
                                            last_success
                                    }


# ============================================================
# 7. CELL COMPLETE
# ============================================================

section(
    "CELL COMPLETE"
)

print(
    "MARKET_CONNECTOR_RESULT:"
)

print(
    MARKET_CONNECTOR_RESULT
)
